In [24]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
pd.set_option("display.max_columns", None)

In [25]:
CSV_PATH = "../openfoodfacts_ueber6prozent.csv.zip"
COLS = ["product_name", "ingredients_analysis_tags", "nova_group", "nutrient_levels_tags"]

df = pd.read_csv(CSV_PATH, sep="\t", usecols=COLS, low_memory=False, on_bad_lines="skip")
df = df.copy()

# Helper: replace junk placeholder strings with NaN
INVALID = {"?", ".", ",", "n-a", "na", "none", "null", "0", "en:null", "en:none"}

def clean_series(s):
    s = s.astype("string").str.strip()
    s = s.replace("", pd.NA)
    s = s.where(~s.str.lower().isin(INVALID), pd.NA)
    s = s.where(~s.str.fullmatch(r"[?,.\-/ ]+", na=False), pd.NA)
    return s

_W = 68

def section_header(title):
    print(f"\n{'═' * _W}")
    print(f"  {title}")
    print(f"{'─' * _W}")

def clean_report(col_name, before, after):
    pct     = after / len(df) * 100
    removed = before - after
    bar     = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<26}  [{bar}]  {pct:5.1f}%   befüllt {after:,}  (entfernt {removed:,})")


# Meilestein 2: Bereinigung der vier Spalten des >6%-Datensatzes
section_header("Meilestein 2  ·  product_name · ingredients_analysis_tags · nova_group · nutrient_levels_tags")

# Product Name — Text: Junk entfernen, trimmen, kleinschreiben, Leerraum normalisieren
product_name_before = df["product_name"].notna().sum()
df["product_name"] = (
    clean_series(df["product_name"])
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)
clean_report("product_name", product_name_before, df["product_name"].notna().sum())

# Ingredients Analysis Tags — kommaseparierte en:-Tags; Junk entfernen, kleinschreiben, Kommas normalisieren
iat_before = df["ingredients_analysis_tags"].notna().sum()
df["ingredients_analysis_tags"] = (
    clean_series(df["ingredients_analysis_tags"])
    .str.lower()
    .str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("ingredients_analysis_tags", iat_before, df["ingredients_analysis_tags"].notna().sum())

# NOVA Group — numerisch erzwingen, nur gültige Gruppen 1-4 behalten (nullable Int)
nova_before = df["nova_group"].notna().sum()
_nova = pd.to_numeric(df["nova_group"], errors="coerce")
df["nova_group"] = _nova.where(_nova.isin([1, 2, 3, 4])).astype("Int64")
clean_report("nova_group", nova_before, df["nova_group"].notna().sum())

# Nutrient Levels Tags — wie ingredients_analysis_tags behandeln
nlt_before = df["nutrient_levels_tags"].notna().sum()
df["nutrient_levels_tags"] = (
    clean_series(df["nutrient_levels_tags"])
    .str.lower()
    .str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("nutrient_levels_tags", nlt_before, df["nutrient_levels_tags"].notna().sum())

print(f"\n{'─' * _W}")
print(f"  Zwischenstand: {df.shape[1]} Spalten gesamt  |  {len(df):,} Zeilen")
print(f"{'─' * _W}")



════════════════════════════════════════════════════════════════════
  Meilestein 2  ·  product_name · ingredients_analysis_tags · nova_group · nutrient_levels_tags
────────────────────────────────────────────────────────────────────
  product_name                [██████████████████░░]   92.6%   befüllt 4,168,836  (entfernt 164)
  ingredients_analysis_tags   [██████░░░░░░░░░░░░░░]   30.1%   befüllt 1,356,120  (entfernt 0)
  nova_group                  [█████░░░░░░░░░░░░░░░]   25.1%   befüllt 1,128,990  (entfernt 1)
  nutrient_levels_tags        [██████░░░░░░░░░░░░░░]   33.8%   befüllt 1,520,527  (entfernt 0)

────────────────────────────────────────────────────────────────────
  Zwischenstand: 4 Spalten gesamt  |  4,501,223 Zeilen
────────────────────────────────────────────────────────────────────


In [26]:
# Begriffs-Normalisierung product_name: häufige FR/IT/ES/DE-Lebensmittelwörter -> Englisch
# (Regex + Wörterbuch, keine echte Übersetzung — unbekannte Wörter bleiben unverändert)
GLOSSAR = {
    # --- Französisch ---
    "lait": "milk", "chocolat": "chocolate", "sucre": "sugar", "farine": "flour",
    "huile": "oil", "beurre": "butter", "oeuf": "egg", "oeufs": "eggs", "oignon": "onion",
    "fromage": "cheese", "pain": "bread", "eau": "water", "jus": "juice", "creme": "cream",
    "crème": "cream", "poulet": "chicken", "boeuf": "beef", "porc": "pork", "poisson": "fish",
    "viande": "meat", "fruits": "fruits", "fruit": "fruit", "legumes": "vegetables",
    "légumes": "vegetables", "legume": "vegetable", "miel": "honey", "sel": "salt",
    "poivre": "pepper", "tomate": "tomato", "pomme": "apple", "fraise": "strawberry",
    "framboise": "raspberry", "vanille": "vanilla", "cafe": "coffee", "café": "coffee",
    "thé": "tea", "vin": "wine", "biscuit": "biscuit", "gateau": "cake", "confiture": "jam",
    "soupe": "soup", "riz": "rice", "jambon": "ham", "blanc": "white", "noir": "black",
    "citron": "lemon", "canard": "duck", "noix": "nuts", "saumon": "salmon", "foie": "liver",
    "fumé": "smoked", "saveur": "flavour", "salade": "salad", "sirop": "syrup", "pur": "pure",
    "cuit": "cooked", "filet": "fillet", "filets": "fillets",
    "sans": "without", "avec": "with", "bio": "organic", "nature": "plain", "entier": "whole",
    # --- Italienisch ---
    "latte": "milk", "cioccolato": "chocolate", "zucchero": "sugar", "farina": "flour",
    "olio": "oil", "burro": "butter", "uovo": "egg", "uova": "eggs", "formaggio": "cheese",
    "pane": "bread", "acqua": "water", "succo": "juice", "crema": "cream", "pollo": "chicken",
    "manzo": "beef", "maiale": "pork", "pesce": "fish", "carne": "meat", "frutta": "fruit",
    "verdura": "vegetables", "miele": "honey", "sale": "salt", "pepe": "pepper", "riso": "rice",
    "caffe": "coffee", "vino": "wine", "biscotti": "biscuits", "torta": "cake", "senza": "without",
    "con": "with", "integrale": "whole", "oliva": "olive",
    # --- Spanisch ---
    "leche": "milk", "azucar": "sugar", "harina": "flour", "aceite": "oil", "mantequilla": "butter",
    "queso": "cheese", "huevo": "egg", "huevos": "eggs", "agua": "water", "zumo": "juice",
    "crema": "cream", "carne": "meat", "pescado": "fish", "fruta": "fruit", "verduras": "vegetables",
    "miel": "honey", "sal": "salt", "pimienta": "pepper", "arroz": "rice", "cafe": "coffee",
    "vino": "wine", "galletas": "biscuits", "pastel": "cake", "sin": "without", "con": "with",
    "sabor": "flavour",
    # --- Deutsch ---
    "milch": "milk", "schokolade": "chocolate", "zucker": "sugar", "mehl": "flour",
    "butter": "butter", "kase": "cheese", "käse": "cheese", "ei": "egg", "eier": "eggs",
    "wasser": "water", "saft": "juice", "sahne": "cream", "huhn": "chicken", "rind": "beef",
    "schwein": "pork", "fisch": "fish", "fleisch": "meat", "obst": "fruit", "gemuse": "vegetables",
    "gemüse": "vegetables", "honig": "honey", "salz": "salt", "pfeffer": "pepper", "reis": "rice",
    "kaffee": "coffee", "wein": "wine", "kuchen": "cake", "ohne": "without", "mit": "with",
    "knoblauch": "garlic", "zwiebel": "onion", "zwiebeln": "onions", "paprika": "pepper", "weiss": "white",

    # --- mehrsprachig (FR/IT/ES) ---
    "cacao": "cocoa", "coco": "coconut",
}

# Ein kompiliertes Regex mit Wortgrenzen; längere Begriffe zuerst (greedy bei Überschneidung)
muster = re.compile(
    r"\b(" + "|".join(sorted(map(re.escape, GLOSSAR), key=len, reverse=True)) + r")\b"
)

def ersetze(m):
    return GLOSSAR[m.group(0)]

section_header("Begriffs-Normalisierung product_name  (FR/IT/ES/DE -> EN, Regex + Glossar)")

# nur einzigartige Namen ersetzen, dann zurückmappen (schnell, vektorisiert)
vorher_na = df["product_name"].isna().sum()
namen = df["product_name"].dropna().drop_duplicates()
ersetzt = namen.str.replace(muster, ersetze, regex=True)
mapping = dict(zip(namen, ersetzt))
geaendert = int((namen != ersetzt).sum())

df["product_name"] = df["product_name"].map(mapping).astype("string").fillna(df["product_name"])

print(f"  Namen mit ersetzten Begriffen: {geaendert:,} von {len(namen):,} einzigartigen")
print(f"  NaN vorher/nachher: {vorher_na:,} / {df['product_name'].isna().sum():,} (darf nicht steigen)")

# Stichprobe: nur Namen, in denen tatsächlich etwas ersetzt wurde
beispiele = [(o, n) for o, n in zip(namen, ersetzt) if o != n][:25]
pd.DataFrame(beispiele, columns=["vorher", "nachher (normalisiert)"])


════════════════════════════════════════════════════════════════════
  Begriffs-Normalisierung product_name  (FR/IT/ES/DE -> EN, Regex + Glossar)
────────────────────────────────────────────────────────────────────
  Namen mit ersetzten Begriffen: 465,772 von 2,369,981 einzigartigen
  NaN vorher/nachher: 332,387 / 332,387 (darf nicht steigen)


,vorher,nachher (normalisiert)
0,confiture extra citron de menton,jam extra lemon de menton
1,granola bio le chocolaté,granola organic le chocolaté
2,nesquik moins de sucre,nesquik moins de sugar
3,croccante con pistacchi e miele,croccante with pistacchi e honey
4,lindt vollmilch schokolade,lindt vollmilch chocolate
5,confiture de grenade aux fruits secs rihana,jam de grenade aux fruits secs rihana
6,fondants citron,fondants lemon
7,pâte à tartiner chocolatée sans allergènes,pâte à tartiner chocolatée without allergènes
8,genoise chocolat bijou,genoise chocolate bijou
9,"poulet entier noir ""le 88 jours""","chicken whole black ""le 88 jours"""
